# Stock Viewer — обозреватель акций с виджетами

Полный график цены акции с подключаемыми виджетами:
- **Event Study** — CAR-кривая, метрики, выделенные зоны событийного/оценочного окон
- **Объём торгов** — полная история объёма

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.path.dirname(''), '..'))

import numpy as np
import pandas as pd

from core.event_study import EventStudy
from core.models import DividendEvent
from core.dividend_data_provider import load_dividends
from core.market_data_provider import load_risk_free_rate, load_market_index
from core.stock_data_provider import get_log_returns, get_stock_data

events_list = load_dividends()
all_tickers = sorted({ev.ticker for ev in events_list})

prices = {t: get_log_returns(t) for t in all_tickers}
market = load_market_index()
rf = load_risk_free_rate()

studies = {t: EventStudy(stock=prices[t]) for t in all_tickers}

stock_data = {}
for ticker in all_tickers:
    df = get_stock_data(ticker, normalized=True)
    df = df[['DATE', 'CLOSE', 'VOL']].copy()
    df['DATE'] = pd.to_datetime(df['DATE'])
    df = df.set_index('DATE').sort_index()
    stock_data[ticker] = df

events_by_ticker = {}
for ev in events_list:
    events_by_ticker.setdefault(ev.ticker, []).append(ev)

print(f'Тикеров: {len(all_tickers)}, Событий: {len(events_list)}')

C:\Users\Ruslan\anaconda3\envs\data-core\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Тикеров: 11, Событий: 110


In [2]:
import socket
import uuid
from datetime import date, timedelta

import plotly.graph_objects as go
from dash import Dash, html, dcc, Input, Output, State, no_update

WEEKDAYS_RU = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']


def _find_free_port(min_port=10001):
    port = min_port
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('localhost', port)) != 0:
                return port
        port += 1


def _event_label(ev: DividendEvent) -> str:
    return f"Анонс дивидендов ({ev.event_date.strftime('%d.%m.%Y')}, {ev.dividend:.2f} руб/акция)"


def _event_value(ev: DividendEvent) -> str:
    return f"{ev.ticker}_{ev.event_date.isoformat()}"


def _card(title, value, tooltip_text, color='#333'):
    """Карточка метрики с CSS tooltip."""
    tooltip_span = html.Span(
        children=[
            ' \u24d8',
            html.Span(
                tooltip_text,
                className='metric-tooltip-text',
                style={
                    'visibility': 'hidden', 'opacity': '0',
                    'position': 'absolute', 'zIndex': '10',
                    'bottom': '120%', 'left': '50%', 'transform': 'translateX(-50%)',
                    'backgroundColor': '#333', 'color': '#fff',
                    'padding': '8px 12px', 'borderRadius': '6px',
                    'fontSize': '12px', 'lineHeight': '1.4',
                    'width': '240px', 'textAlign': 'left',
                    'boxShadow': '0 2px 8px rgba(0,0,0,0.2)',
                    'transition': 'opacity 0.2s',
                    'pointerEvents': 'none',
                },
            ),
        ],
        className='metric-tooltip-trigger',
        style={'fontSize': '12px', 'color': '#bbb', 'cursor': 'help',
               'position': 'relative', 'display': 'inline-block'},
    )
    return html.Div([
        html.Div([
            html.Span(title, style={'fontSize': '13px', 'color': '#888'}),
            tooltip_span,
        ], style={'marginBottom': '4px'}),
        html.Div(value, style={'fontSize': '22px', 'fontWeight': 'bold', 'color': color}),
    ], style={
        'flex': '1', 'textAlign': 'center', 'padding': '12px 8px',
        'backgroundColor': 'white', 'borderRadius': '8px',
        'boxShadow': '0 1px 3px rgba(0,0,0,0.12)',
    })


def _compute_event_metrics(
    ar: np.ndarray,
    ew_before: int,
    vol_days: list,
    vol_values: list,
) -> tuple[float, float]:
    """Вычисляет коэффициенты волатильности и объёма (после/до t=0)."""
    ar_before = ar[:ew_before] if ew_before > 0 else np.array([])
    ar_after = ar[ew_before:] if ew_before < len(ar) else np.array([])

    vb = float(np.std(ar_before) * 100) if len(ar_before) > 1 else 0
    va = float(np.std(ar_after) * 100) if len(ar_after) > 1 else 0
    vol_ratio = round(va / vb, 2) if vb > 0 else float('nan')

    if vol_days:
        mb = np.mean([v for d, v in zip(vol_days, vol_values) if d < 0]) if any(d < 0 for d in vol_days) else 0
        ma = np.mean([v for d, v in zip(vol_days, vol_values) if d > 0]) if any(d > 0 for d in vol_days) else 0
        volume_ratio = round(ma / mb, 2) if mb > 0 else float('nan')
    else:
        volume_ratio = float('nan')

    return vol_ratio, volume_ratio


def _resolve_event_dates(
    ticker: str,
    event_date: date,
    ew_before: int,
    ew_after: int,
    estimation_window: int,
) -> dict:
    """Маппит day offsets на реальные даты для event study зон."""
    sd = stock_data[ticker]
    trading_days = sd.index.sort_values()
    t0 = pd.Timestamp(event_date)
    idx0 = int(trading_days.searchsorted(t0, side='left'))

    day_dates = {}
    for d in range(-ew_before, ew_after + 1):
        pos = idx0 + d
        if 0 <= pos < len(trading_days):
            day_dates[d] = trading_days[pos]

    ew_start_pos = idx0 - ew_before
    est_end_pos = ew_start_pos - 1
    est_start_pos = est_end_pos - estimation_window + 1

    result = {'t0': t0.isoformat(), 'day_dates': {str(k): v.isoformat() for k, v in day_dates.items()}}

    if 0 <= ew_start_pos < len(trading_days):
        result['ev_start'] = trading_days[ew_start_pos].isoformat()
    ev_end_pos = idx0 + ew_after
    if 0 <= ev_end_pos < len(trading_days):
        result['ev_end'] = trading_days[ev_end_pos].isoformat()
    if est_start_pos >= 0 and 0 <= est_end_pos < len(trading_days):
        result['est_start'] = trading_days[est_start_pos].isoformat()
        result['est_end'] = trading_days[est_end_pos].isoformat()

    return result


def _add_event_zones(fig: go.Figure, es_data: dict) -> None:
    """Добавляет зоны событийного/оценочного окон и t=0 (на всю высоту)."""
    est_start = es_data.get('est_start')
    est_end = es_data.get('est_end')
    ev_start = es_data.get('ev_start')
    ev_end = es_data.get('ev_end')
    t0 = es_data.get('t0')

    if est_start and est_end:
        fig.add_shape(
            type='rect', x0=est_start, x1=est_end, y0=0, y1=1,
            yref='paper', xref='x',
            fillcolor='rgba(150,150,150,0.1)', line_width=0,
        )
    if ev_start and ev_end:
        fig.add_shape(
            type='rect', x0=ev_start, x1=ev_end, y0=0, y1=1,
            yref='paper', xref='x',
            fillcolor='rgba(100,149,237,0.12)', line_width=0,
        )
    if t0:
        fig.add_shape(
            type='line', x0=t0, x1=t0, y0=0, y1=1,
            yref='paper', xref='x',
            line=dict(color='crimson', width=1.5, dash='dash'),
        )
        fig.add_annotation(
            x=t0, y=1, yref='paper', text='t=0',
            showarrow=False, font=dict(color='crimson', size=11),
            xanchor='left', yanchor='bottom',
        )


RANGE_BUTTONS = [
    dict(count=1, label='1М', step='month', stepmode='backward'),
    dict(count=3, label='3М', step='month', stepmode='backward'),
    dict(count=6, label='6М', step='month', stepmode='backward'),
    dict(count=1, label='1Г', step='year', stepmode='backward'),
    dict(count=5, label='5Г', step='year', stepmode='backward'),
    dict(label='Всё', step='all'),
]


def _build_main_figure(
    ticker: str,
    es_data: dict | None = None,
    show_es: bool = False,
    show_vol: bool = False,
) -> go.Figure:
    """Строит график: цена + объём (отдельная панель) + [CAR панель сверху]."""
    sd = stock_data[ticker]
    has_car = show_es and es_data and 'car_pct' in es_data

    fig = go.Figure()

    # --- Domain-ы осей Y (без перекрытия) ---
    if has_car and show_vol:
        car_domain = [0.80, 1.0]
        price_domain = [0.22, 0.76]
        vol_domain = [0.0, 0.18]
    elif has_car:
        car_domain = [0.77, 1.0]
        price_domain = [0.0, 0.73]
        vol_domain = None
    elif show_vol:
        car_domain = None
        price_domain = [0.22, 1.0]
        vol_domain = [0.0, 0.18]
    else:
        car_domain = None
        price_domain = [0.0, 1.0]
        vol_domain = None

    # --- Цена ---
    price_hover = [
        f"<b>{d.strftime('%d.%m.%Y')} ({WEEKDAYS_RU[d.weekday()]})</b><br>"
        f"Цена: {sd.loc[d, 'CLOSE']:,.2f} руб."
        for d in sd.index
    ]
    fig.add_trace(go.Scatter(
        x=sd.index, y=sd['CLOSE'], mode='lines',
        line=dict(color='#8B5CF6', width=1.5),
        name='Цена', hovertext=price_hover, hoverinfo='text',
        yaxis='y',
    ))

    # --- Объём (отдельная панель снизу) ---
    if show_vol and vol_domain:
        vol_hover = [
            f"<b>{d.strftime('%d.%m.%Y')} ({WEEKDAYS_RU[d.weekday()]})</b><br>"
            f"Объём: {sd.loc[d, 'VOL']:,.0f}"
            for d in sd.index
        ]
        fig.add_trace(go.Scatter(
            x=sd.index, y=sd['VOL'],
            fill='tozeroy', fillcolor='rgba(100,149,237,0.25)',
            line=dict(color='rgba(100,149,237,0.5)', width=0.5),
            name='Объём', hovertext=vol_hover, hoverinfo='text',
            showlegend=False, yaxis='y2',
        ))

    # --- CAR панель ---
    if has_car and car_domain:
        dd = es_data['day_dates']
        car_data = es_data['car_pct']
        ci_upper = es_data.get('ci_upper', [])
        ci_lower = es_data.get('ci_lower', [])

        sorted_keys = sorted(dd.keys(), key=int)
        dates = [pd.Timestamp(dd[k]) for k in sorted_keys]
        n = min(len(dates), len(car_data))

        if ci_upper and ci_lower:
            fig.add_trace(go.Scatter(
                x=dates[:n] + dates[:n][::-1],
                y=ci_upper[:n] + ci_lower[:n][::-1],
                fill='toself', fillcolor='rgba(100,149,237,0.15)',
                line=dict(width=0), name='95% ДИ', hoverinfo='skip',
                yaxis='y3',
            ))

        car_hover = [f"CAR: {car_data[i]:+.3f}%" for i in range(n)]
        fig.add_trace(go.Scatter(
            x=dates[:n], y=car_data[:n], mode='lines+markers',
            line=dict(color='steelblue', width=2.5), marker=dict(size=4),
            name='CAR, %', hovertext=car_hover, hoverinfo='text',
            yaxis='y3',
        ))

    # --- Зоны ES ---
    if has_car and es_data:
        _add_event_zones(fig, es_data)

    # --- Разделители между панелями ---
    for domain in [car_domain, price_domain, vol_domain]:
        if domain and domain[0] > 0.01:
            fig.add_shape(
                type='line', x0=0, x1=1, y0=domain[0] - 0.01, y1=domain[0] - 0.01,
                xref='paper', yref='paper',
                line=dict(color='rgba(180,180,180,0.5)', width=1),
            )

    # --- Layout: X range + Y range по видимому окну ---
    sd_last = sd.index[-1]
    sd_1y_ago = sd_last - timedelta(days=365)
    x_range = [sd_1y_ago, sd_last]

    if has_car and es_data:
        est_start = es_data.get('est_start')
        ev_end = es_data.get('ev_end')
        if est_start and ev_end:
            x_range = [pd.Timestamp(est_start) - timedelta(days=30),
                       pd.Timestamp(ev_end) + timedelta(days=30)]

    # Y range по видимому X окну (чтобы rangeslider не сплющивал)
    visible = sd.loc[x_range[0]:x_range[1]]
    if len(visible) > 0:
        y_min = visible['CLOSE'].min()
        y_max = visible['CLOSE'].max()
        y_pad = (y_max - y_min) * 0.05
        price_y_range = [y_min - y_pad, y_max + y_pad]
    else:
        price_y_range = None

    xaxis_cfg = dict(
        rangeselector=dict(buttons=RANGE_BUTTONS, x=0, y=1.0, xanchor='left'),
        rangeslider=dict(visible=True, thickness=0.04),
        type='date',
        range=[x_range[0].isoformat(), x_range[1].isoformat()],
    )

    layout = dict(
        template='plotly_white',
        height=650,
        margin=dict(l=50, r=30, t=40, b=20),
        showlegend=True,
        legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='right', x=1),
        hovermode='x',
        uirevision=ticker,
        xaxis=xaxis_cfg,
        yaxis=dict(
            title='Цена, руб.', domain=price_domain,
            range=price_y_range, fixedrange=False,
        ),
    )

    if show_vol and vol_domain:
        layout['yaxis2'] = dict(
            title='Объём', domain=vol_domain,
            anchor='x', showgrid=False,
            autorange=True, fixedrange=False,
        )

    if has_car and car_domain:
        layout['yaxis3'] = dict(
            title='CAR, %', domain=car_domain,
            anchor='x', showgrid=True,
            autorange=True, fixedrange=False,
        )

    fig.update_layout(**layout)
    return fig

In [3]:
# === Dash-приложение ===

default_ticker = all_tickers[0]
default_events = events_by_ticker.get(default_ticker, [])

app = Dash(f'stock_viewer_{uuid.uuid4().hex[:8]}')

app.index_string = '''<!DOCTYPE html>
<html>
    <head>
        {%metas%}
        <title>{%title%}</title>
        {%favicon%}
        {%css%}
        <style>
        .metric-tooltip-trigger:hover .metric-tooltip-text {
            visibility: visible !important;
            opacity: 1 !important;
        }
        </style>
    </head>
    <body>
        {%app_entry%}
        <footer>
            {%config%}
            {%scripts%}
            {%renderer%}
        </footer>
    </body>
</html>'''

app.layout = html.Div([
    html.H2('Stock Viewer', style={'marginBottom': '15px'}),

    html.Div([
        html.Label('Компания'),
        dcc.Dropdown(
            id='ticker-dropdown',
            options=[{'label': t, 'value': t} for t in all_tickers],
            value=default_ticker,
            clearable=False, style={'width': '200px'},
        ),
    ], style={'marginBottom': '15px'}),

    dcc.Checklist(
        id='widget-toggles',
        options=[
            {'label': ' Event Study', 'value': 'event_study'},
            {'label': ' Объём торгов', 'value': 'volume'},
        ],
        value=['volume'],
        inline=True,
        style={'marginBottom': '15px', 'fontSize': '15px'},
    ),

    # --- Контролы Event Study ---
    html.Div(id='es-controls-panel', children=[
        html.Div([
            html.Div([
                html.Label('Событие'),
                dcc.Dropdown(
                    id='event-dropdown',
                    options=[{'label': _event_label(ev), 'value': _event_value(ev)} for ev in default_events],
                    value=_event_value(default_events[0]) if default_events else None,
                    clearable=False, style={'width': '100%'},
                ),
            ], style={'flex': '3', 'marginRight': '10px'}),
            html.Div([
                html.Label('Модель'),
                dcc.Dropdown(
                    id='model-dropdown',
                    options=[
                        {'label': 'Mean Adjusted', 'value': 'mean_adjusted'},
                        {'label': 'Market Model', 'value': 'market_model'},
                        {'label': 'CAPM', 'value': 'capm'},
                    ],
                    value='market_model', clearable=False, style={'width': '100%'},
                ),
            ], style={'flex': '1'}),
        ], style={'display': 'flex', 'marginBottom': '10px'}),

        html.Div([
            html.Div([
                html.Label('Дней ДО'),
                dcc.Slider(id='ew-before', min=1, max=40, step=1, value=10,
                           marks={i: str(i) for i in [1, 5, 10, 20, 30, 40]},
                           tooltip={'placement': 'bottom', 'always_visible': True}),
            ], style={'flex': '1', 'marginRight': '15px'}),
            html.Div([
                html.Label('Дней ПОСЛЕ'),
                dcc.Slider(id='ew-after', min=1, max=40, step=1, value=10,
                           marks={i: str(i) for i in [1, 5, 10, 20, 30, 40]},
                           tooltip={'placement': 'bottom', 'always_visible': True}),
            ], style={'flex': '1', 'marginRight': '15px'}),
            html.Div([
                html.Label('Оценочное окно'),
                dcc.Slider(id='est-length', min=30, max=500, step=10, value=200,
                           marks={i: str(i) for i in [30, 100, 200, 300, 400, 500]},
                           tooltip={'placement': 'bottom', 'always_visible': True}),
            ], style={'flex': '1'}),
        ], style={'display': 'flex', 'marginBottom': '15px'}),

        html.Div([
            html.Button('\u2190', id='prev-event-btn', n_clicks=0,
                        style={'padding': '6px 14px', 'fontSize': '16px', 'cursor': 'pointer', 'marginRight': '8px'}),
            html.Button('Рассчитать', id='calc-button', n_clicks=0,
                        style={'backgroundColor': '#4CAF50', 'color': 'white', 'border': 'none',
                               'padding': '8px 24px', 'fontSize': '15px', 'cursor': 'pointer', 'borderRadius': '4px'}),
            html.Button('\u2192', id='next-event-btn', n_clicks=0,
                        style={'padding': '6px 14px', 'fontSize': '16px', 'cursor': 'pointer', 'marginLeft': '8px'}),
            html.Span(id='nav-label', style={'marginLeft': '12px', 'fontSize': '14px', 'color': '#888'}),
        ], style={'textAlign': 'center', 'marginBottom': '10px'}),
    ], style={'display': 'none', 'padding': '12px', 'backgroundColor': '#f9f9f9',
              'borderRadius': '8px', 'marginBottom': '15px'}),

    # --- Метрики ---
    html.Div(id='metrics-cards', style={'display': 'none', 'gap': '15px', 'marginBottom': '15px'}),

    # --- Основной график (цена + CAR overlay + объём) ---
    dcc.Graph(id='main-graph'),

    # --- Хранилище ---
    dcc.Store(id='es-result-store'),
    html.Div(id='error-msg', style={'color': 'red', 'textAlign': 'center', 'marginTop': '10px'}),
])


# ---------------------------------------------------------------------------
# Callbacks
# ---------------------------------------------------------------------------

@app.callback(
    Output('es-controls-panel', 'style'),
    Output('metrics-cards', 'style'),
    Input('widget-toggles', 'value'),
)
def toggle_widgets(widgets):
    es_on = 'event_study' in (widgets or [])
    es_style = {'display': 'block', 'padding': '12px', 'backgroundColor': '#f9f9f9',
                'borderRadius': '8px', 'marginBottom': '15px'} if es_on else {'display': 'none'}
    metrics_style = {'display': 'flex', 'gap': '15px', 'marginBottom': '15px'} if es_on else {'display': 'none'}
    return es_style, metrics_style


@app.callback(
    Output('event-dropdown', 'options'),
    Output('event-dropdown', 'value'),
    Input('ticker-dropdown', 'value'),
)
def update_events_dropdown(ticker):
    evs = events_by_ticker.get(ticker, [])
    options = [{'label': _event_label(ev), 'value': _event_value(ev)} for ev in evs]
    value = _event_value(evs[0]) if evs else None
    return options, value


@app.callback(
    Output('event-dropdown', 'value', allow_duplicate=True),
    Input('prev-event-btn', 'n_clicks'),
    Input('next-event-btn', 'n_clicks'),
    State('ticker-dropdown', 'value'),
    State('event-dropdown', 'value'),
    prevent_initial_call=True,
)
def navigate_event(prev_clicks, next_clicks, ticker, current_val):
    from dash import ctx
    if not current_val:
        return no_update
    evs = events_by_ticker.get(ticker, [])
    if not evs:
        return no_update
    values = [_event_value(ev) for ev in evs]
    try:
        idx = values.index(current_val)
    except ValueError:
        return no_update
    if ctx.triggered_id == 'prev-event-btn':
        idx = max(0, idx - 1)
    elif ctx.triggered_id == 'next-event-btn':
        idx = min(len(values) - 1, idx + 1)
    return values[idx]


@app.callback(
    Output('nav-label', 'children'),
    Input('event-dropdown', 'value'),
    State('ticker-dropdown', 'value'),
)
def update_nav_label(current_val, ticker):
    evs = events_by_ticker.get(ticker, [])
    if not evs or not current_val:
        return ''
    values = [_event_value(ev) for ev in evs]
    try:
        idx = values.index(current_val)
    except ValueError:
        return ''
    return f'{idx + 1} / {len(values)}'


@app.callback(
    Output('main-graph', 'figure'),
    Input('ticker-dropdown', 'value'),
    Input('es-result-store', 'data'),
    Input('widget-toggles', 'value'),
)
def render_main_chart(ticker, es_data, widgets):
    widgets = widgets or []
    show_es = 'event_study' in widgets
    show_vol = 'volume' in widgets
    return _build_main_figure(ticker, es_data if show_es else None, show_es, show_vol)


@app.callback(
    Output('metrics-cards', 'children'),
    Output('es-result-store', 'data'),
    Output('error-msg', 'children'),
    Input('calc-button', 'n_clicks'),
    Input('event-dropdown', 'value'),
    State('ticker-dropdown', 'value'),
    State('model-dropdown', 'value'),
    State('ew-before', 'value'),
    State('ew-after', 'value'),
    State('est-length', 'value'),
    prevent_initial_call=True,
)
def calculate_event_study(n_clicks, event_val, ticker, model_type, ew_before, ew_after, est_length):
    if not event_val:
        return no_update, no_update, 'Выберите событие'

    parts = event_val.split('_', 1)
    target_date = date.fromisoformat(parts[1])

    event = next((ev for ev in events_by_ticker.get(ticker, []) if ev.event_date == target_date), None)
    if event is None:
        return no_update, no_update, f'Событие не найдено: {ticker} {target_date}'

    extra = {}
    if model_type in ('market_model', 'capm'):
        extra['market'] = market
    if model_type == 'capm':
        extra['rf'] = rf

    result = studies[ticker].analyze(
        event_date=event.event_date,
        model=model_type,
        event_window=(-ew_before, ew_after),
        estimation_window=est_length,
        **extra,
    )
    if result is None:
        return no_update, no_update, 'Недостаточно данных. Попробуйте уменьшить окна.'

    dates_info = _resolve_event_dates(ticker, event.event_date, ew_before, ew_after, est_length)

    ar = result.ar
    car_pct = (np.cumsum(ar) * 100).tolist()
    sigma = result.estimation_std
    ci_upper = [2 * sigma * np.sqrt(i) * 100 for i in range(1, result.n_days + 1)]
    ci_lower = [-v for v in ci_upper]

    dates_info['ar'] = ar
    dates_info['car_pct'] = car_pct
    dates_info['ci_upper'] = ci_upper
    dates_info['ci_lower'] = ci_lower

    # Метрики
    day_dates_iso = dates_info['day_dates']
    sorted_keys = sorted(day_dates_iso.keys(), key=int)
    vol_days = [int(k) for k in sorted_keys]
    sd = stock_data.get(ticker)
    vol_values = []
    if sd is not None:
        for k in sorted_keys:
            dt = pd.Timestamp(day_dates_iso[k])
            vol_values.append(sd.loc[dt, 'VOL'] if dt in sd.index else 0)

    vol_ratio, volume_ratio = _compute_event_metrics(np.array(ar), ew_before, vol_days, vol_values)

    car_total = result.car * 100
    car_color = '#2e7d32' if car_total >= 0 else '#c62828'
    cards = [
        _card('CAR, %', f'{car_total:+.3f}%',
              'Сумма дневных аномальных доходностей за все дни окна.', car_color),
        _card('Коэф. волатильности', f'{vol_ratio}',
              '> 1 — рынок нервничал после события, < 1 — успокоился.'),
        _card('Коэф. объёма', f'{volume_ratio}',
              '> 1 — объём торгов вырос после события, < 1 — упал.'),
        _card('Дней в окне', f'{result.n_days}',
              'Фактическое количество торговых дней в событийном окне.'),
    ]

    return cards, dates_info, ''


port = _find_free_port()
print(f'Dash запущен на http://127.0.0.1:{port}/')
app.run(port=port, jupyter_mode='inline')

Dash запущен на http://127.0.0.1:10002/
